# Linear Elasticity in 2D

This example demonstrates the solution of a simple plane stress problem using LowLevelFEM.

The problem is solved in two different ways:

1. Using the built-in elasticity solver.
2. Using the weak-form DSL interface.

The results are compared with the analytical solution.

In [1]:
using LowLevelFEM

## Geometry and Material

A unit square domain is discretized using a structured mesh.

The material is assumed to be homogeneous, isotropic, and linearly elastic under plane stress conditions.

In [2]:
structured_rect_mesh()

mat = Material("body", E=2e5, ν=0.3)
prob = Problem([mat], type=:PlaneStress)

Problem("structured_rect", :PlaneStress, 2, 2, Material[Material("body", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 121, LowLevelFEM.Geometry("", "", 0, 0, nothing, nothing, nothing, nothing), :u, :f)

## Boundary Conditions and Loading

The left edge is fixed in the horizontal direction, and the bottom edge is fixed in the vertical direction.

A uniform tensile load is applied to the right edge.

In [3]:
ld = load("right", fx=1)

bc1 = displacementConstraint("left", ux=0)
bc2 = displacementConstraint("bottom", uy=0)

BoundaryCondition("bottom", nothing, Dict{Symbol, Union{Function, Number, ScalarField}}(:uy => 0))

## Solution Using the Built-in Elasticity Solver

In [4]:
u = solveDisplacement(prob, load=[ld], support=[bc1, bc2])

nodal VectorField
[0.0; 0.0; … ; 4.500000000000012e-6; -1.3500000000000078e-6;;]

The strain and stress fields are computed from the displacement solution.

In [5]:
ε = solveStrain(u)
σ = solveStress(u)

elementwise TensorField
[[0.9999999999999996; 0.0; … ; 0.0; 0.0;;], [0.9999999999999993; -4.8867285418517323e-17; … ; 0.0; 0.0;;], [0.9999999999999992; 0.0; … ; 0.0; 0.0;;], [1.0000000000000002; 1.303127611160462e-16; … ; 0.0; 0.0;;], [1.000000000000001; -1.303127611160462e-16; … ; 0.0; 0.0;;], [1.0000000000000013; 0.0; … ; 0.0; 0.0;;], [1.0000000000000007; -2.606255222320924e-16; … ; 0.0; 0.0;;], [1.0000000000000013; 0.0; … ; 0.0; 0.0;;], [1.0000000000000016; -3.909382833481386e-16; … ; 0.0; 0.0;;], [1.0000000000000018; -1.303127611160462e-16; … ; 0.0; 0.0;;]  …  [1.0000000000000004; 0.0; … ; 0.0; 0.0;;], [1.000000000000001; 5.853023728079444e-16; … ; 0.0; 0.0;;], [1.0000000000000029; 1.755907118423833e-15; … ; 0.0; 0.0;;], [0.9999999999999991; 0.0; … ; 0.0; 0.0;;], [1.0000000000000022; 0.0; … ; 0.0; 0.0;;], [0.9999999999999979; 1.0425020889283696e-15; … ; 0.0; 0.0;;], [1.0000000000000022; 0.0; … ; 0.0; 0.0;;], [0.9999999999999963; -1.0425020889283696e-15; … ; 0.0; 0.0;;], [0.99999999

In [6]:
probe(u, 1, 1, 0)

2-element Vector{Float64}:
  5.0e-6
 -1.5e-6

In [7]:
probe(σ, 0.5, 0.5, 0)

3×3 Matrix{Float64}:
 1.0   0.0  0.0
 0.0  -0.0  0.0
 0.0   0.0  0.0

## Post-processing

Displacements, strains, and stresses can be visualized directly in the Gmsh post-processor.

In [8]:
showDoFResults(u, name="u vec", visible=true)
showDoFResults(u, name="u", factor=4e4)
showDoFResults(u, :x, name="ux")
showDoFResults(u, :y, name="uy")

showStrainResults(ε, :x, name="εx")
showStrainResults(ε, :y, name="εy")
showStrainResults(ε, :xy, name="γxy")

showStressResults(σ, name="σ_eqv")
showStressResults(σ, name="σ_eqv", smooth=true)
showStressResults(σ, :x, name="σx")
showStressResults(σ, :y, name="σy")
showStressResults(σ, :xy, name="τxy")

11

# The Same Problem Using the DSL Interface

The following section solves exactly the same problem by assembling the weak form directly.

This approach provides greater flexibility and is particularly useful when implementing custom formulations.

In [9]:
Pu = Problem([mat], type=:VectorField, dim=2, field=:u, rhs_field=:f)

Problem("structured_rect", :VectorField, 2, 2, Material[Material("body", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 121, LowLevelFEM.Geometry("", "", 0, 0, nothing, nothing, nothing, nothing), :u, :f)

## Constitutive Matrix

For plane stress conditions, the constitutive matrix is

$$
D=\frac{E}{1-\nu^2}
\begin{bmatrix}
1 & \nu & 0\\
\nu & 1 & 0\\
0 & 0 & \frac{1-\nu}{2}
\end{bmatrix}
$$

In [10]:
E = mat.E
ν = mat.ν
D = E / (1 - ν^2) * [1 ν 0; ν 1 0; 0 0 (1-ν)/2]

3×3 Matrix{Float64}:
     2.1978e5  65934.1           0.0
 65934.1           2.1978e5      0.0
     0.0           0.0       76923.1

## Weak Form Assembly

The stiffness matrix is assembled directly from the weak form

$$
K=\int_{\Omega}\varepsilon^T D \varepsilon\, d\Omega
$$

using the symbolic operators provided by LowLevelFEM.

In [11]:
K = ∫(SymGrad(Pu) ⋅ D ⋅ SymGrad(Pu))

sparse([1, 2, 9, 10, 79, 80, 81, 82, 1, 2  …  47, 48, 221, 222, 223, 224, 239, 240, 241, 242], [1, 1, 1, 1, 1, 1, 1, 1, 2, 2  …  242, 242, 242, 242, 242, 242, 242, 242, 242, 242], [98901.09890109889, 35714.28571428572, -60439.56043956043, -2747.2527472527518, 10989.010989010983, 2747.252747252749, -49450.54945054945, -35714.28571428571, 35714.28571428572, 98901.09890109889  …  35714.28571428571, -49450.54945054944, -35714.28571428571, -49450.549450549406, 1.1823431123048067e-11, 21978.02197802202, -1.864464138634503e-11, -120879.1208791209, -2.1827872842550278e-11, 395604.39560439566], 242, 242)

In [12]:
f = ∫(Pu ⋅ [1, 0], Γ="right")

nodal VectorField
[0.0; 0.0; … ; 0.0; 0.0;;]

In [13]:
bc3 = BoundaryCondition("left", ux=0)
bc4 = BoundaryCondition("bottom", uy=0)

BoundaryCondition("bottom", nothing, Dict{Symbol, Union{Function, Number, ScalarField}}(:uy => 0))

## Solution of the Discrete System

In [14]:
u2 = solveField(K, f, support=[bc3, bc4])

nodal VectorField
[0.0; 0.0; … ; 4.499999999999992e-6; -1.3500000000000021e-6;;]

## Post-processing of the DSL Solution

The strain and stress components are reconstructed explicitly from the displacement field.

In [15]:
εx = ∂x(u2[1])
εy = ∂y(u2[2])
γxy = ∂y(u2[1]) + ∂x(u2[2])

ε2 = [εx, εy, γxy]

3-element Vector{ScalarField}:
 ScalarField([[4.9999999999999996e-6; 4.9999999999999996e-6; 5.0e-6; 5.0e-6;;], [5.0e-6; 5.0e-6; 4.999999999999999e-6; 4.999999999999999e-6;;], [4.999999999999999e-6; 4.9999999999999996e-6; 4.999999999999997e-6; 4.999999999999998e-6;;], [4.999999999999998e-6; 4.999999999999997e-6; 4.999999999999996e-6; 4.999999999999996e-6;;], [4.999999999999996e-6; 4.999999999999996e-6; 4.999999999999995e-6; 4.999999999999995e-6;;], [4.999999999999995e-6; 4.999999999999995e-6; 4.999999999999995e-6; 4.999999999999995e-6;;], [4.999999999999995e-6; 4.999999999999995e-6; 4.999999999999994e-6; 4.999999999999994e-6;;], [4.999999999999994e-6; 4.999999999999994e-6; 4.999999999999992e-6; 4.999999999999992e-6;;], [4.999999999999992e-6; 4.999999999999992e-6; 4.999999999999989e-6; 4.999999999999989e-6;;], [4.999999999999989e-6; 4.999999999999989e-6; 4.999999999999985e-6; 4.999999999999985e-6;;]  …  [4.999999999999993e-6; 4.999999999999993e-6; 5.000000000000006e-6; 5.000000000000006e

In [16]:
σ2 = D * ε2

3-element Vector{ScalarField}:
 ScalarField([[0.9999999999999993; 0.9999999999999993; 0.9999999999999996; 0.9999999999999996;;], [0.9999999999999996; 0.9999999999999996; 0.9999999999999991; 0.9999999999999991;;], [0.9999999999999991; 0.9999999999999993; 0.9999999999999989; 0.9999999999999989;;], [0.999999999999999; 0.9999999999999989; 0.9999999999999987; 0.9999999999999988;;], [0.9999999999999989; 0.9999999999999988; 0.9999999999999986; 0.9999999999999987;;], [0.9999999999999986; 0.9999999999999987; 0.9999999999999987; 0.9999999999999986;;], [0.9999999999999988; 0.9999999999999987; 0.9999999999999982; 0.9999999999999983;;], [0.999999999999998; 0.9999999999999986; 0.9999999999999983; 0.9999999999999978;;], [0.9999999999999983; 0.9999999999999982; 0.9999999999999976; 0.9999999999999977;;], [0.999999999999998; 0.9999999999999977; 0.9999999999999968; 0.9999999999999971;;]  …  [0.9999999999999981; 0.9999999999999982; 1.0000000000000013; 1.0000000000000013;;], [1.0000000000000013; 1.00000000

In [17]:
σx = σ2[1]
σy = σ2[2]
τxy = σ2[3]

elementwise ScalarField
[[0.0; 1.623384252879259e-16; 1.6289095139505776e-16; 0.0;;], [0.0; -1.303127611160462e-16; -6.51563805580231e-17; 6.51563805580231e-17;;], [6.51563805580231e-17; -1.2368244783046377e-16; 6.51563805580231e-17; 1.954691416740693e-16;;], [1.954691416740693e-16; 3.7767765677697213e-16; 6.51563805580231e-17; 0.0;;], [0.0; -5.96311194867048e-17; -1.954691416740693e-16; -1.303127611160462e-16;;], [-1.303127611160462e-16; -1.303127611160462e-16; 0.0; 0.0;;], [0.0; -1.303127611160462e-16; -2.606255222320924e-16; -1.303127611160462e-16;;], [-1.303127611160462e-16; -1.954691416740693e-16; 4.560946639061618e-16; 5.212510444641848e-16;;], [5.212510444641848e-16; 3.257819027901155e-16; 1.954691416740693e-16; 3.909382833481386e-16;;], [3.909382833481386e-16; 6.51563805580231e-17; -3.257819027901155e-16; 0.0;;]  …  [0.0; 5.212510444641848e-16; 6.189856153012195e-16; 9.773457083703465e-17;;], [-4.2351647362715017e-16; -4.2351647362715017e-16; -3.5836009306912706e-16; -3.5836009

In [18]:
showDoFResults(u2, name="u (DSL)")

showElementResults(εx, name="εx (DSL)")
showElementResults(εy, name="εy (DSL)")
showElementResults(γxy, name="γxy (DSL)")

σeqv = √(((σx - σy) * (σx - σy) + σx * σx + σy * σy + 6τxy * τxy) / 2)

showElementResults(σeqv, name="σ_eqv (DSL)")
showElementResults(σx, name="σx (DSL)")
showElementResults(σy, name="σy (DSL)")
showElementResults(τxy, name="τxy (DSL)")

19

# Comparison with the Analytical Solution

For this simple uniaxial tension problem, the exact solution is known.

The displacement at the right edge can therefore be compared directly with the finite element results.

In [19]:
σx_ex = 1
εx_ex = σx_ex / E
u_max = εx_ex * 1

5.0e-6

In [20]:
probe(u[1], 1, 1, 0)

5.0e-6

In [21]:
probe(u2[1], 1, 1, 0)

5.0e-6

## Open the Post-processor

Open the Gmsh post-processor to inspect the computed fields.

In [22]:
openPostProcessor()

XOpenIM() failed
Fontconfig warning: using without calling FcInit()


## Summary

This example demonstrated:

- solution of a plane stress problem using the built-in elasticity solver,
- solution of the same problem using the weak-form DSL,
- computation of strains and stresses,
- comparison with an analytical solution.

Both approaches produce the same result, while the DSL formulation provides direct access to the weak form.